In [11]:
import re
import string 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.preprocessing import FunctionTransformer
from bs4 import BeautifulSoup

In [12]:
from pathlib import Path

TRAIN_PATH = Path("../data/processed/train.csv")

train_df = pd.read_csv(TRAIN_PATH)

X_train = train_df["review"]
y_train = train_df["sentiment"]

print("Training reviews:", len(X_train))
print("Training labels:", y_train.value_counts())

Training reviews: 40006
Training labels: sentiment
positive    20004
negative    20002
Name: count, dtype: int64


In [13]:
def strip_html_batch(texts):
    return [
        BeautifulSoup(
            "" if pd.isna(text) else str(text),
            "html.parser"
        ).get_text(separator=" ")
        for text in texts
    ]


baseline = Pipeline([
    (
        "html",
        FunctionTransformer(
            strip_html_batch,
            validate=False
        )
    ),
    (
        "vectorizer",
        CountVectorizer()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

print("Baseline pipeline created successfully!")

Baseline pipeline created successfully!


In [14]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("Fold accuracies:", np.round(scores, 4))
print(f"Mean CV accuracy: {scores.mean():.4f}")
print(f"Standard deviation: {scores.std():.4f}")

Fold accuracies: [0.8865 0.8859 0.8913 0.8855 0.8895]
Mean CV accuracy: 0.8877
Standard deviation: 0.0023


In [15]:
def lowercase_batch(texts):
    return [text.lower() for text in texts]


def remove_punctuation_batch(texts):
    return [
        re.sub(r"[^\w\s]", " ", text)
        for text in texts
    ]


# Keep negation words to preserve sentiment
custom_stop_words = list(
    ENGLISH_STOP_WORDS - {"not", "no", "never"}
)


def remove_stopwords_batch(texts):
    cleaned = []

    for text in texts:
        words = text.split()
        words = [
            word for word in words
            if word.lower() not in custom_stop_words
        ]
        cleaned.append(" ".join(words))

    return cleaned


print("Preprocessing functions ready!")

Preprocessing functions ready!


In [16]:
from snowballstemmer import stemmer

english_stemmer = stemmer("english")

def stem_text_batch(texts):
    return [
        " ".join(english_stemmer.stemWords(str(text).lower().split()))
        for text in texts
    ]

print(stem_text_batch([
    "The movies were amazing and entertaining",
    "The performances were disappointing"
]))

['the movi were amaz and entertain', 'the perform were disappoint']


In [17]:
variants = {
    "Raw text": [],
    "HTML stripping": [strip_html_batch],
    "Lowercasing": [strip_html_batch, lowercase_batch],
    "Punctuation removal": [
        strip_html_batch,
        lowercase_batch,
        remove_punctuation_batch
    ],
    "Stop-word removal": [
        strip_html_batch,
        lowercase_batch,
        remove_punctuation_batch,
        remove_stopwords_batch
    ]
}

results = []

for variant_name, preprocessing_functions in variants.items():
    steps = []

    for step_number, preprocessing_function in enumerate(
        preprocessing_functions
    ):
        steps.append((
            f"preprocessing_{step_number}",
            FunctionTransformer(
                preprocessing_function,
                validate=False
            )
        ))

    steps.extend([
        ("vectorizer", CountVectorizer()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ])

    model = Pipeline(steps)

    variant_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    results.append({
        "Variant": variant_name,
        "Mean CV Accuracy": variant_scores.mean(),
        "Std Dev": variant_scores.std()
    })

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))


            Variant  Mean CV Accuracy  Std Dev
           Raw text          0.887742 0.002300
     HTML stripping          0.887742 0.002254
        Lowercasing          0.887742 0.002254
Punctuation removal          0.887742 0.002254
  Stop-word removal          0.883468 0.002731
